In [9]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import numpy as np
import pandas as pd

def calculate_snr(original, noisy):
    """Calculate Signal-to-Noise Ratio (SNR)"""
    if hasattr(original, 'toarray'):
        original = original.toarray()
    if hasattr(noisy, 'toarray'):
        noisy = noisy.toarray()
    
    signal_power = np.var(original)
    noise = original - noisy
    noise_power = np.var(noise)
    
    if noise_power == 0:
        return float('inf')
    
    return 10 * np.log10(signal_power / noise_power)

# Create Excel writer
output_file = "SNR_Analysis_Results.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for combination_id in [1, 2, 3, 4]:
        try:
            # Read data
            adata_RNA = sc.read_h5ad(f'../../../Data/Noise_Combination_{combination_id}/Combination{combination_id}_RNA.h5ad')
            adata_ADT = sc.read_h5ad(f'../../../Data/Noise_Combination_{combination_id}/Combination{combination_id}_Protein.h5ad')

            RNA_DATA = [adata_RNA.X, adata_RNA.obsm['level_1'], adata_RNA.obsm['level_2'], adata_RNA.obsm['level_3']]
            ADT_DATA = [adata_ADT.X, adata_ADT.obsm['level_1'], adata_ADT.obsm['level_2'], adata_ADT.obsm['level_3']]

            # Calculate SNR
            combo_data = []
            
            # RNA data
            for i in range(1, 4):
                snr_value = calculate_snr(RNA_DATA[0], RNA_DATA[i])
                combo_data.append({
                    'Data_Modality': 'RNA',
                    'Noise_Level': f'level_{i}',
                    'SNR_dB': round(snr_value, 2)  # 保留2位小数
                })
            
            # ADT data
            for i in range(1, 4):
                snr_value = calculate_snr(ADT_DATA[0], ADT_DATA[i])
                combo_data.append({
                    'Data_Modality': 'ADT', 
                    'Noise_Level': f'level_{i}',
                    'SNR_dB': round(snr_value, 2)  # 保留2位小数
                })
            
            # Create pivot table
            combo_df = pd.DataFrame(combo_data)
            pivot_df = (
                combo_df
                .pivot_table(
                    index='Data_Modality',
                    columns='Noise_Level', 
                    values='SNR_dB',
                    aggfunc='first'
                )
                .reindex(["RNA", "ADT"])
                .reset_index()
            )
            pivot_df.columns.name = None
            pivot_df = pivot_df.rename(columns={'Data_Modality': 'Data Modality'})
            
            # Write to Excel
            sheet_name = f'Combination_{combination_id}'
            pivot_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        except FileNotFoundError:
            print(f"Warning: Data files for Combination {combination_id} not found, skipping...")
        except Exception as e:
            print(f"Error processing Combination {combination_id}: {e}")